In [1]:
pip install pandas numpy scikit-learn nltk

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import numpy as np
import re
import nltk

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from nltk.corpus import stopwords
from sklearn.metrics import accuracy_score, classification_report
from sklearn.feature_extraction.text import TfidfVectorizer

In [3]:
fake = pd.read_csv("Fake.csv")
true = pd.read_csv("True.csv")
fake.head()

,title,text,subject,date
0,Donald Trump Sends Out Embarrassing New Year’...,Donald Trump just couldn t wish all Americans ...,News,"December 31, 2017"
1,Drunk Bragging Trump Staffer Started Russian ...,House Intelligence Committee Chairman Devin Nu...,News,"December 31, 2017"
2,Sheriff David Clarke Becomes An Internet Joke...,"On Friday, it was revealed that former Milwauk...",News,"December 30, 2017"
3,Trump Is So Obsessed He Even Has Obama’s Name...,"On Christmas day, Donald Trump announced that ...",News,"December 29, 2017"
4,Pope Francis Just Called Out Donald Trump Dur...,Pope Francis used his annual Christmas Day mes...,News,"December 25, 2017"


In [4]:
true.head()

,title,text,subject,date
0,"As U.S. budget fight looms, Republicans flip t...",WASHINGTON (Reuters) - The head of a conservat...,politicsNews,"December 31, 2017"
1,U.S. military to accept transgender recruits o...,WASHINGTON (Reuters) - Transgender people will...,politicsNews,"December 29, 2017"
2,Senior U.S. Republican senator: 'Let Mr. Muell...,WASHINGTON (Reuters) - The special counsel inv...,politicsNews,"December 31, 2017"
3,FBI Russia probe helped by Australian diplomat...,WASHINGTON (Reuters) - Trump campaign adviser ...,politicsNews,"December 30, 2017"
4,Trump wants Postal Service to charge 'much mor...,SEATTLE/WASHINGTON (Reuters) - President Donal...,politicsNews,"December 29, 2017"


In [5]:
fake["label"] = 1
true["label"] = 0

In [6]:
df = pd.concat([fake, true], axis=0)

df = df.sample(frac=1).reset_index(drop=True)

df.head()

,title,text,subject,date,label
0,NANCY PELOSI’S Latest Over-The-Top Claim: Trum...,Nancy Pelosi s latest claim is over the top:In...,left-news,"Apr 25, 2017",1
1,OBAMA UNLEASHES HUNDREDS Of Intelligence Agent...,We all know by now that the biggest threat to ...,left-news,"Nov 8, 2016",1
2,"White Security Guard Shoots Himself, Blames N...",On Tuesday night St. Catherine University secu...,News,"September 15, 2017",1
3,U.S. senators urge Trump to take tough line on...,WASHINGTON (Reuters) - A bipartisan group of 2...,politicsNews,"December 8, 2016",0
4,IAEA chief calls for clarity on disputed secti...,neVIENNA (Reuters) - The U.N. nuclear watchdog...,worldnews,"September 26, 2017",0


In [10]:
nltk.download("stopwords")

stop_words = set(stopwords.words("english"))

def preprocess(text):
    
    text = str(text)              # convert to string
    text = text.lower()           # lowercase
    text = re.sub('[^a-zA-Z]', ' ', text)   # remove numbers/punctuation
    
    words = text.split()
    
    words = [word for word in words if word not in stop_words]
    
    return " ".join(words)

[nltk_data] Downloading package stopwords to C:\Users\Aryan
[nltk_data]     Harke\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [11]:
df = df.dropna()

df["title"] = df["title"].astype(str)
df["text"] = df["text"].astype(str)

df["content"] = df["title"] + " " + df["text"]

df["content"] = df["content"].apply(preprocess)

In [12]:
X=df["content"]
y=df["label"]

In [13]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [14]:
X_train = X_train.astype(str)
X_test = X_test.astype(str)

In [15]:
vectorizer = TfidfVectorizer(max_features=5000)

X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

In [16]:
model = LogisticRegression(max_iter=1000)

model.fit(X_train_vec, y_train)

LogisticRegression(max_iter=1000)

In [20]:
y_pred = model.predict(X_test_vec)

In [21]:
print("Accuracy:", accuracy_score(y_test, y_pred))

print(classification_report(y_test, y_pred))

Accuracy: 0.9861915367483296
              precision    recall  f1-score   support

           0       0.98      0.99      0.99      4232
           1       0.99      0.98      0.99      4748

    accuracy                           0.99      8980
   macro avg       0.99      0.99      0.99      8980
weighted avg       0.99      0.99      0.99      8980



In [22]:
import joblib

joblib.dump(model, "fake_news_model.pkl")
joblib.dump(vectorizer, "vectorizer.pkl")

['vectorizer.pkl']